# Rebuild and test our CPU heading model

Download the 28 frozen World Food Programme (WFP) reports, verify their hashes, train the original tree classifiers, select the confidence settings on validation reports, and predict headings. No private archive, original weights, API key or GPU is required. Native PDF text is required; this is not an OCR model.

From the repository root: `uv sync --locked --extra training --extra notebook`, then `uv run --extra training --extra notebook jupyter lab notebooks/rebuild_heading_model.ipynb`. Select the project Python kernel and **Run All**. Full logs and models stay outside Git. Downloads fail if a published PDF no longer matches its recorded checksum.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

from pdf_heading_benchmark.io import REPO

DATA = Path(os.environ.get("PDF_BENCHMARK_DATA", REPO.parent / "pdf-heading-benchmark-data")).resolve()
TRAINING = DATA / "structure-v2"
RUN = DATA / "rebuilds" / datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
MODELS = RUN / "models"
SELECTION = RUN / "precision"
RUN.mkdir(parents=True, exist_ok=False)
os.chdir(REPO)
print("New rebuild created. No historical experiment files will be overwritten.")

## 1. Download and prepare the original examples

The frozen manifest contains 20 training reports, four validation reports and four previously inspected regression reports. Preparation requires every source line in the annotations to match the PDF extraction. Cached PDFs are checksum-verified before reuse. A mismatch stops the run; it never silently changes the corpus or labels.

In [ ]:
with (RUN / "prepare.log").open("w") as log:
    subprocess.run(
        [sys.executable, "scripts/structure/prepare.py", "--download", "--data", str(TRAINING)],
        check=True,
        stdout=log,
        stderr=subprocess.STDOUT,
    )
dataset = json.loads((TRAINING / "dataset.json").read_text())
print(f"Prepared {len(dataset['documents'])} checksum-verified reports.")

## 2. Fit the classifiers

The existing training program fits all five classifiers needed to compare the three original variants. Each uses 200 Extra Trees, maximum depth 16, minimum leaf size 2 and seed 42. Only the 20 training reports fit the trees; the four validation reports select a variant. The deployed context variant needs three classifiers: line joining, heading classification and level relationships.

In [ ]:
with (RUN / "train.log").open("w") as log:
    subprocess.run(
        [sys.executable, "scripts/structure/train.py", "--data", str(TRAINING), "--output", str(MODELS)],
        check=True,
        stdout=log,
        stderr=subprocess.STDOUT,
    )
training = json.loads((MODELS / "selection.json").read_text())
print("Selected variant:", training["selected"])
print("Training examples:", training["training_examples"])
print("Classifier fitting seconds:", round(sum(training["fit_seconds"].values()), 3))

## 3. Select the confidence settings

Run the same 36-setting validation search as the published experiment. It maximizes joint heading-and-level F0.5 subject to the historical heading-only precision constraint. This constraint is part of reproducing the selection procedure, not a PASS/FAIL gate in the main report. Model hashes are regenerated for this build. If no setting qualifies, stop; do not borrow the historical setting or tune on regression results.

In [ ]:
with (RUN / "select.log").open("w") as log:
    subprocess.run(
        [
            sys.executable,
            "scripts/structure/optimize_precision.py",
            "select",
            "--data",
            str(TRAINING),
            "--run",
            str(MODELS),
            "--output",
            str(SELECTION),
        ],
        check=True,
        stdout=log,
        stderr=subprocess.STDOUT,
    )
selected = json.loads((SELECTION / "selection.json").read_text())["selected"]
if selected is None:
    raise ValueError(
        "No qualifying validation setting. Inspect select.log; do not use test results to choose one."
    )
print("Settings:", selected["settings"])
print("Validation joint F0.5:", round(selected["score"]["joint_f05"], 4))

## 4. Evaluate without changing the selected settings

Download the four historical benchmark PDFs, then run the existing regression program. It scores the previously inspected 38-heading set and all 77 historical reference headings. These are regression checks, not a new independent evaluation. The historical run parses all 523 pages to preserve document context. Its recorded duration is a measured full-document time, not a measured 32-page time.

In [ ]:
from pdf_heading_benchmark.corpus import prepare as download_benchmark

download_benchmark(DATA)
with (RUN / "regression.log").open("w") as log:
    subprocess.run(
        [
            sys.executable,
            "scripts/structure/optimize_precision.py",
            "regression",
            "--data",
            str(TRAINING),
            "--run",
            str(MODELS),
            "--output",
            str(SELECTION),
        ],
        check=True,
        stdout=log,
        stderr=subprocess.STDOUT,
    )
regression = json.loads((SELECTION / "regression.json").read_text())
historical = json.loads((SELECTION / "historical.json").read_text())
counts = {
    name: sum(row[name] for row in historical["scores"])
    for name in ("gold", "predicted", "matched", "correct_level")
}
print("38-heading regression F0.5:", round(regression["results"]["selected"]["joint_f05"], 4))
print("Historical counts:", counts)
print("Measured seconds for all 523 pages:", round(sum(row["seconds"] for row in historical["timings"]), 3))

## 5. Predict on a PDF

The default is one downloaded benchmark report. Change `PDF` to your own PDF with selectable text, and use a fresh `OUTPUT` folder for each run. Markdown preserves the native text and marks predicted headings. JSON includes the heading text, physical PDF page and level. The separately reported parser time excludes model loading and output writes.

In [ ]:
manifest = json.loads((REPO / "configs/corpus.json").read_text())
PDF = DATA / "corpus" / f"{manifest[0]['id']}.pdf"  # Replace with Path("/path/to/your.pdf").
OUTPUT = RUN / "prediction"
with (RUN / "predict.log").open("w") as log:
    subprocess.run(
        [
            sys.executable,
            "scripts/runners/run_precision.py",
            str(PDF),
            str(OUTPUT),
            "--run",
            str(MODELS),
            "--selection",
            str(SELECTION),
        ],
        check=True,
        stdout=log,
        stderr=subprocess.STDOUT,
    )
headings = json.loads((OUTPUT / "structure_precision.headings.json").read_text())
print(f"Wrote Markdown and JSON with {len(headings)} predicted headings.")
for heading in headings[:8]:
    print(f"Page {heading['page']}, H{heading['level']}: {heading['title']}")
print("Output folder:", OUTPUT)

## What this run establishes

A successful Run All verifies public download, annotation preparation, training, validation selection, regression and prediction using this environment. Compare the recorded results with the published evidence before claiming the same model quality. Model-file hashes can differ because the JSON includes measured fitting time; the regression predictions are the behavioral check. A rebuild does not establish generalization beyond these WFP documents.